In [0]:
from pyspark.sql.functions import *


In [0]:
silver_path = "/Volumes/workspace/default/retail_data/silver/"

orders = spark.read.parquet(silver_path + "orders_clean.parquet")
products = spark.read.parquet(silver_path + "products_clean.parquet")
customers = spark.read.parquet(silver_path + "customers_secure.parquet")

In [0]:
#Sales summary per product
sales_summary = orders.join(products, "product_id") \
    .groupBy("product_id", "product_name", "category") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity_sold"),
        round(sum("total_amount"), 2).alias("total_revenue"),
        round(avg("total_amount"), 2).alias("avg_order_value")
    )

In [0]:
#Summary per customer token
customer_summary = orders.join(customers, "customer_token") \
    .groupBy("customer_token", "age_band", "city", "state") \
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("total_amount"), 2).alias("total_spent"),
        round(avg("total_amount"), 2).alias("avg_order_value")
    )

In [0]:
#Product summary per category
product_summary = orders.join(products, "product_id") \
    .groupBy("category") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_units_sold"),
        round(sum("total_amount"), 2).alias("total_revenue")
    )

In [0]:
#Revenue per category
category_revenue = orders.join(products, "product_id") \
    .groupBy("category") \
    .agg(round(sum("total_amount"), 2).alias("category_revenue"))

#Revenue per month
monthly_revenue = orders.withColumn("order_month", date_format("order_date", "yyyy-MM")) \
    .groupBy("order_month") \
    .agg(round(sum("total_amount"), 2).alias("monthly_revenue"))

In [0]:
#Save to gold layer
gold_path = "/Volumes/workspace/default/retail_data/gold/"

sales_summary.write.mode("overwrite").parquet(gold_path + "sales_summary.parquet")
customer_summary.write.mode("overwrite").parquet(gold_path + "customer_summary.parquet")
product_summary.write.mode("overwrite").parquet(gold_path + "product_summary.parquet")
category_revenue.write.mode("overwrite").parquet(gold_path + "category_revenue.parquet")
monthly_revenue.write.mode("overwrite").parquet(gold_path + "monthly_revenue.parquet")